# WTA Match Predictor — Modelo ML

Entrenamiento, evaluación y guardado del modelo. Requiere `historico_partidos.csv` generado por el notebook de EDA.

**Salidas de este notebook:**
- `src/model/gbx_v3.model` — mejor modelo XGBoost
- `src/model/random_v3.model` — mejor modelo Random Forest
- `src/model/voting.model` — ensemble Voting (XGB + RF)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, auc, log_loss
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier

In [ ]:
# Rutas relativas al repositorio (ejecutar desde src/notebooks/)
DATA_DIR  = os.path.join('..', 'data')
MODEL_DIR = os.path.join('..', 'model')

## 2. Carga de datos

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'historico_partidos.csv'), parse_dates=['date'])
print(f'Partidos cargados: {len(df)}')
print(df.info())

## 3. Train / Test split

Split temporal: el test es siempre el período más reciente para simular condiciones reales de predicción. El orden cronológico garantiza que el modelo nunca entrena con información futura.

In [ ]:
corte    = '2025-01-01'
df_train = df[df['date'] < corte].copy()
df_test  = df[df['date'] >= corte].copy()

print(f'Train: {len(df_train)} partidos ({df_train["date"].min().year}–{df_train["date"].max().year})')
print(f'Test:  {len(df_test)} partidos ({df_test["date"].min().year}–{df_test["date"].max().year})')
print(f'Test representa el {len(df_test)/len(df):.1%} del dataset total')

df_test.columns

## 4. Tratamiento de outliers

Capeamos `rank_diff` usando los percentiles del train — nunca del test para evitar data leakage.

In [ ]:
limite_superior = df_train['rank_diff'].quantile(0.90)
limite_inferior  = df_train['rank_diff'].quantile(0.10)

df_train['rank_diff'] = df_train['rank_diff'].clip(lower=limite_inferior, upper=limite_superior)
df_test['rank_diff']  = df_test['rank_diff'].clip(lower=limite_inferior, upper=limite_superior)

print(f'Límite inferior (p10%):  {limite_inferior}')
print(f'Límite superior (p90%): {limite_superior}')

## 5. Preparar X e y

Se excluyen `match_id`, `date` (identificadores sin valor predictivo), `target` (la etiqueta) y `odd_1`/`odd_2` (las probabilidades de las casas de apuestas — se usan solo como baseline de comparación, nunca como input del modelo).

Se probó incluir y excluir `tournament_type` — el modelo da mejor resultado sin ella.

In [ ]:
# Versión final: sin tournament_type (probada con ella — empeoraba)
cols_excluir = ['match_id', 'date', 'target', 'odd_1', 'odd_2', 'tournament_type']

X_train = df_train.drop(columns=cols_excluir)
X_test  = df_test.drop(columns=cols_excluir)
y_train = (df_train['target'] == 1).astype(int)
y_test  = (df_test['target'] == 1).astype(int)

print(f'Features finales: {X_train.columns.tolist()}')
print(f'Shape train: {X_train.shape} | test: {X_test.shape}')

In [ ]:
# Verificar tipos y nulos
print(X_train.dtypes)
print('\nNulos:')
print(X_train.isnull().sum())

## 6. Pipeline y preprocesador

In [ ]:
# Las columnas categóricas reciben OneHotEncoder; las numéricas, StandardScaler
# tournament_type se probó como categórica pero se descartó (ver sección 5)
cat_cols = ['surface', 'round']
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

## 7. Modelos

Se entrenaron tres modelos con búsqueda de hiperparámetros por GridSearchCV con validación cruzada temporal (cv=5):
- **Random Forest**
- **XGBoost**
- **Voting Classifier** (ensemble soft de los dos anteriores)

Los mejores modelos de cada iteración se guardaron en `model/` con sufijo de versión para trazabilidad.

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

rf_params = {
    'classifier__max_depth':    [2, 3, 5],
    'classifier__max_features': ['sqrt', 'log2', 0.3, 0.5],
    'classifier__n_estimators': [100, 200]
}

random_best = GridSearchCV(estimator=pipe_rf, param_grid=rf_params, cv=5, n_jobs=-1)
random_best.fit(X_train, y_train)

print(f'Mejor score RF (CV): {random_best.best_score_:.2%}')
print(f'Mejores parámetros:  {random_best.best_params_}')

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
pipe_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(eval_metric='logloss', random_state=42))
])

xgb_params = {
    'classifier__max_depth':        [3, 4, 5, 6],
    'classifier__n_estimators':     [100, 200],
    'classifier__learning_rate':    [0.05, 0.1, 0.2],
    'classifier__subsample':        [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

xgb_best = GridSearchCV(estimator=pipe_xgb, param_grid=xgb_params, cv=5, n_jobs=-1)
xgb_best.fit(X_train, y_train)

print(f'Mejor score XGB (CV): {xgb_best.best_score_:.2%}')
print(f'Mejores parámetros:   {xgb_best.best_params_}')

In [ ]:
# ── Voting Classifier (ensemble soft) ─────────────────────────────────────────
# Carga los mejores modelos de la versión v3 y los combina en un ensemble de votación ponderada
xgb_modelo = pickle.load(open(os.path.join(MODEL_DIR, 'gbx_v3.model'), 'rb'))
rf_modelo  = pickle.load(open(os.path.join(MODEL_DIR, 'random_v3.model'), 'rb'))

voting = VotingClassifier(
    estimators=[
        ('xgb', xgb_modelo),
        ('rf',  rf_modelo)
    ],
    voting='soft'
)

voting.fit(X_train, y_train)

## 8. Evaluación y comparativa

### Métricas usadas
- **Accuracy**: porcentaje de predicciones correctas.
- **AUC-ROC**: capacidad discriminativa del modelo (independiente del umbral de clasificación).
- **Log Loss**: penaliza las predicciones confiadas y erróneas — es importante si las probabilidades si van a usar directamente (predicciones o apuestas)

### Baselines de referencia
- **Casas de apuestas**: predicen la jugadora con menor odd (mayor probabilidad implícita).
- **Ranking**: predice siempre que gana la jugadora con mejor ranking.

In [ ]:
# DataFrame acumulador de resultados de todos los experimentos
resultados = pd.DataFrame({
    'modelo':           pd.Series(dtype='str'),
    'parametros':       pd.Series(dtype='str'),
    'accuracy_modelo':  pd.Series(dtype='float'),
    'accuracy_casas':   pd.Series(dtype='float'),
    'accuracy_ranking': pd.Series(dtype='float'),
    'AUC_modelo':       pd.Series(dtype='float'),
    'AUC_casas':        pd.Series(dtype='float'),
    'logloss_modelo':   pd.Series(dtype='float'),
})

In [ ]:
def evaluar_modelo(nombre, modelo_fit, X_test, y_test, df_test, df):
    y_pred  = modelo_fit.predict(X_test)
    y_proba = modelo_fit.predict_proba(X_test)[:, 1]

    acc_modelo = accuracy_score(y_test, y_pred)

    # Odds del test (se usan solo para comparar, no para entrenar)
    df_test_con_odds = df_test[['match_id']].copy()
    df_test_con_odds = df_test_con_odds.merge(df[['match_id', 'odd_1', 'odd_2']],
                                               on='match_id', how='left')

    # Accuracy baseline casas de apuestas
    prob_1 = df_test_con_odds['odd_1'].values
    prob_2 = df_test_con_odds['odd_2'].values
    mask   = ~np.isnan(prob_1) & ~np.isnan(prob_2)
    pred_casas = (prob_1[mask] > prob_2[mask]).astype(int)
    acc_casas  = accuracy_score(y_test[mask], pred_casas)

    # Accuracy baseline ranking
    pred_ranking = (df_test['rank_diff'] < 0).astype(int)
    acc_ranking  = accuracy_score(y_test, pred_ranking)

    # AUC modelo
    auc_modelo = roc_auc_score(y_test, y_proba)

    # AUC casas
    prob_casas = df_test_con_odds['odd_1'].values
    mask_auc   = ~np.isnan(prob_casas)
    auc_casas  = roc_auc_score(y_test[mask_auc], prob_casas[mask_auc])

    # Log Loss
    logloss_modelo = log_loss(y_test, y_proba)

    print(f'--- {nombre} ---')
    print(f'AUC modelo: {auc_modelo:.4f} | AUC casas: {auc_casas:.4f}')
    print(f'Acc modelo: {acc_modelo:.4f} | Acc casas: {acc_casas:.4f} | Acc ranking: {acc_ranking:.4f}')
    print(f'LogLoss:    {logloss_modelo:.4f}')

    try:
        params = modelo_fit.best_params_
    except:
        params = {}

    nueva_fila = {
        'modelo':           nombre,
        'parametros':       str(params),
        'accuracy_modelo':  round(acc_modelo, 4),
        'accuracy_casas':   round(acc_casas, 4),
        'accuracy_ranking': round(acc_ranking, 4),
        'AUC_modelo':       round(auc_modelo, 4),
        'AUC_casas':        round(auc_casas, 4),
        'logloss_modelo':   round(logloss_modelo, 4)
    }
    return pd.concat([resultados, pd.DataFrame([nueva_fila])], ignore_index=True)

In [ ]:
# Diagnóstico de overfitting: comparar métricas en train y test
for name, model in [("RF", random_best), ("XGB", xgb_best)]:
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test,  model.predict(X_test))
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    test_auc  = roc_auc_score(y_test,  model.predict_proba(X_test)[:, 1])
    train_ll  = log_loss(y_train, model.predict_proba(X_train))
    test_ll   = log_loss(y_test,  model.predict_proba(X_test))

    print(f"{name} | Acc train: {train_acc:.3f} test: {test_acc:.3f} "
          f"| AUC train: {train_auc:.3f} test: {test_auc:.3f} "
          f"| LogLoss train: {train_ll:.3f} test: {test_ll:.3f}")

In [ ]:
resultados = evaluar_modelo('Random Forest',  random_best, X_test, y_test, df_test, df)
resultados = evaluar_modelo('XGBoost',         xgb_best,    X_test, y_test, df_test, df)
resultados = evaluar_modelo('Voting (XGB+RF)', voting,      X_test, y_test, df_test, df)

In [ ]:
resultados

## 8.1 Trazabilidad de iteraciones

Se guardaron los resultados de cada experimento en CSVs versionados para poder comparar el impacto de cada cambio. A continuación se muestra la comparativa de todas las versiones.

| Versión | Cambio principal |
|---------|-----------------|
| v1 | Features iniciales — todas incluidas (`tournament_type` dentro) |
| v2 | Quitar `tournament_type` + `max_features='sqrt'` en RF |
| v3 | Añadir ELO por superficie y ELO global (sin `inactividad`) |
| v4 | Voting ensemble con los dos mejores de v3 |
| v5 | Verificación: XGBoost sin `tournament_type` (confirmación de v2) |

In [ ]:
resultados1 = pd.read_csv('resultados_features_v1.csv')
resultados2 = pd.read_csv('resultados_features_v2.csv')
resultados3 = pd.read_csv('resultados_features_v3.csv')
resultados4 = pd.read_csv('resultados_features_v4.csv')
resultados5 = pd.read_csv('resultados_features_v5.csv')

todos = pd.concat([resultados1, resultados2, resultados3, resultados4, resultados5], ignore_index=True)
todos

## 8.2 Curva ROC — comparativa de modelos vs casas de apuestas

In [ ]:
xgb_v3    = pickle.load(open(os.path.join(MODEL_DIR, 'gbx_v3.model'), 'rb'))
random_v3 = pickle.load(open(os.path.join(MODEL_DIR, 'random_v3.model'), 'rb'))

y_proba_xgb = xgb_v3.predict_proba(X_test)[:, 1]
y_proba_rf  = random_v3.predict_proba(X_test)[:, 1]

# Probabilidades de las casas alineadas con y_test
df_test_con_odds = df_test[['match_id']].merge(
    df[['match_id', 'odd_1']], on='match_id', how='left'
)
mask_casas    = df_test_con_odds['odd_1'].notna()
y_proba_casas = df_test_con_odds.loc[mask_casas, 'odd_1'].values
y_test_casas  = y_test[mask_casas.values]

# Curvas ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)
fpr_rf,  tpr_rf,  _ = roc_curve(y_test, y_proba_rf)
fpr_cas, tpr_cas, _ = roc_curve(y_test_casas, y_proba_casas)

auc_xgb = auc(fpr_xgb, tpr_xgb)
auc_rf  = auc(fpr_rf,  tpr_rf)
auc_cas = auc(fpr_cas, tpr_cas)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_xgb, tpr_xgb, label=f'XGBoost            (AUC = {auc_xgb:.3f})', color='steelblue')
ax.plot(fpr_rf,  tpr_rf,  label=f'Random Forest      (AUC = {auc_rf:.3f})',  color='darkorange')
ax.plot(fpr_cas, tpr_cas, label=f'Casas de apuestas  (AUC = {auc_cas:.3f})', color='green', linestyle='--')
ax.plot([0, 1], [0, 1],  label='Baseline aleatorio', color='grey', linestyle=':')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curva ROC — Comparativa de modelos')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_comparativa.png', dpi=150)
plt.show()

## 9. Feature importance

In [ ]:
# Cargar los modelos guardados para inspección de importancias
with open(os.path.join(MODEL_DIR, 'gbx_v3.model'), 'rb') as f:
    modeloML = pickle.load(f)

with open(os.path.join(MODEL_DIR, 'random_v3.model'), 'rb') as f:
    modeloML_random = pickle.load(f)

In [ ]:
# Feature importance — XGBoost
fig, ax = plt.subplots(figsize=(10, 8))

feature_names = modeloML.named_steps['preprocessor'].get_feature_names_out()
importancias  = modeloML.named_steps['classifier'].feature_importances_

df_imp = pd.DataFrame({'feature': feature_names, 'importancia': importancias})
df_imp = df_imp.sort_values('importancia', ascending=False).head(15)

sns.barplot(data=df_imp, x='importancia', y='feature', ax=ax)

for bar, valor in zip(ax.patches, df_imp['importancia']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{valor:.1%}', va='center', ha='left', fontsize=9)

ax.set_title('XGBoost — Top 15 features')
ax.set_xlabel('Importancia')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print(df_imp.to_string())

In [ ]:
# Feature importance — Random Forest
fig, ax = plt.subplots(figsize=(10, 8))

feature_names_rf = modeloML_random.named_steps['preprocessor'].get_feature_names_out()
importancias_rf  = modeloML_random.named_steps['classifier'].feature_importances_

df_imp_ran = pd.DataFrame({'feature': feature_names_rf, 'importancia': importancias_rf})
df_imp_ran = df_imp_ran.sort_values('importancia', ascending=False)

sns.barplot(data=df_imp_ran, x='importancia', y='feature', ax=ax)
ax.set_title('Random Forest — Feature importance')
ax.set_xlabel('Importancia')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print(df_imp_ran.to_string())

## 10. Guardar los modelos

Se guarda el mejor estimador de cada GridSearchCV (sin el objeto de búsqueda). El modelo final usado en la app y en la simulación es `gbx_v3.model` (XGBoost).

Referencia del dataset de entrenamiento final:
```
match_id, date, odd_1, odd_2, surface, round, tournament_type,
rank_diff, wins2meses_p1, wins2meses_p2, ratio_superficie_p1, ratio_superficie_p2,
h2h, ratio_ronda_p1, ratio_ronda_p2, experiencia_p1, experiencia_p2,
is_new_p1, is_new_p2, elo_p1, elo_p2, elo_diff,
elo_global_p1, elo_global_p2, elo_global_diff, target
```
(42.046 partidos, 0 nulos)

In [ ]:
with open(os.path.join(MODEL_DIR, 'gbx_v3.model'), 'wb') as f:
    pickle.dump(xgb_best.best_estimator_, f)
print('✓ gbx_v3.model guardado')

with open(os.path.join(MODEL_DIR, 'random_v3.model'), 'wb') as f:
    pickle.dump(random_best.best_estimator_, f)
print('✓ random_v3.model guardado')

with open(os.path.join(MODEL_DIR, 'voting.model'), 'wb') as f:
    pickle.dump(voting, f)
print('✓ voting.model guardado')